# 1. Khởi tạo dữ liệu

In [1]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

# Tạo bảng student
student_data = [
    [1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7],
    [2, "Tran Thi Lan", "Kinh Te", 34, 9.2],
    [3, "Pham Van Nam", "Toan Tin", None, 7.9],
    [4, "Le Thanh Huyen", "Toan Tin", 20, 7.2],
    [5, "Vu Quoc Anh", "May Tinh", 24, 8.0],
    [6, "Dang Thuy Linh", "May Tinh", 24, 5.5],
    [7, "Bui Tien Dung", "Kinh Te", 34, 9.2],
    [8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8],
    [9, "Duong Huu Phuc", "Kinh Te", None, 7.2],
    [10, "Cao Thi Hanh", "May Tinh", None, 7.0],
]

student = pd.DataFrame(student_data, columns=["student_id","name","class","course_id","score"])

# Tạo bảng course
course_data = [
    [12, "Giai tich"],
    [34, "Thong ke"],
    [26, "Tin hoc"]
]

course = pd.DataFrame(course_data, columns=["id","course_name"])

print(student)
print(course)

   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7
1           2       Tran Thi Lan   Kinh Te       34.0    9.2
2           3       Pham Van Nam  Toan Tin        NaN    7.9
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2
4           5        Vu Quoc Anh  May Tinh       24.0    8.0
5           6     Dang Thuy Linh  May Tinh       24.0    5.5
6           7      Bui Tien Dung   Kinh Te       34.0    9.2
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2
9          10       Cao Thi Hanh  May Tinh        NaN    7.0
   id course_name
0  12   Giai tich
1  34    Thong ke
2  26     Tin hoc


# 2. JOIN các kiểu


In [2]:
# INNER JOIN
inner = pd.merge(student, course, left_on="course_id", right_on="id", how="inner")

# LEFT JOIN
left = pd.merge(student, course, left_on="course_id", right_on="id", how="left")

# RIGHT JOIN
right = pd.merge(student, course, left_on="course_id", right_on="id", how="right")

# FULL OUTER JOIN
full = pd.merge(student, course, left_on="course_id", right_on="id", how="outer")

print(inner)
print(left)
print(right)
print(full)

   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te       34.0    9.2  34    Thong ke
   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN         NaN
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN         NaN
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN         NaN
5           6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN         NaN
6           7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8

+ INNER JOIN
Chỉ giữ lại các sinh viên có course_id tồn tại trong bảng course
Kết luận:
→ Dữ liệu “sạch”, loại bỏ các bản ghi thiếu hoặc sai môn học.

+ LEFT JOIN
Giữ toàn bộ sinh viên, kể cả khi không có course hợp lệ
Kết luận:
→ Dùng để phát hiện dữ liệu thiếu (NULL ở course_name).

+ FULL OUTER JOIN
Kết hợp tất cả dữ liệu từ 2 bảng
Kết luận:
→ Cho cái nhìn tổng thể: vừa thấy dữ liệu thiếu ở student, vừa thấy course chưa dùng.

# 3. Cập nhật course_id thiếu & loại bỏ môn không tồn tại

In [3]:
# Lấy danh sách course hợp lệ
valid_ids = course["id"].tolist()

# Điền giá trị thiếu (ví dụ chọn id đầu tiên hợp lệ = 12)
student["course_id"] = student["course_id"].fillna(12)

# Loại bỏ course_id không tồn tại
student = student[student["course_id"].isin(valid_ids)]

print(student)

   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7
1           2       Tran Thi Lan   Kinh Te       34.0    9.2
2           3       Pham Van Nam  Toan Tin       12.0    7.9
6           7      Bui Tien Dung   Kinh Te       34.0    9.2
8           9     Duong Huu Phuc   Kinh Te       12.0    7.2
9          10       Cao Thi Hanh  May Tinh       12.0    7.0


Sau khi xử lý course_id

Kết luận:

Dữ liệu đã được chuẩn hóa
Không còn:
+ course_id bị NULL
+ course_id không tồn tại trong bảng course
→ Đảm bảo tính toàn vẹn dữ liệu (data integrity)

# 4a. Tổng SV + điểm TB theo lớp

In [4]:
class_stats = student.groupby("class").agg(
    total_students=("student_id","count"),
    avg_score=("score","mean")
)

print(class_stats)

          total_students  avg_score
class                              
Kinh Te                3   8.533333
May Tinh               2   6.850000
Toan Tin               1   7.900000


Thống kê theo lớp

Kết luận:

+ Các lớp như Kinh Te có điểm trung bình cao hơn
+ Lớp May Tinh có nhiều sinh viên nhưng điểm phân bố rộng (có cả thấp và cao)

→ Nhận xét:

Có sự chênh lệch chất lượng học tập giữa các lớp

# 4b. Tổng SV + điểm TB theo môn

In [5]:
# JOIN trước
df = pd.merge(student, course, left_on="course_id", right_on="id", how="inner")

course_stats = df.groupby("course_name").agg(
    total_students=("student_id","count"),
    avg_score=("score","mean")
)

print(course_stats)

             total_students  avg_score
course_name                           
Giai tich                 4        7.2
Thong ke                  2        9.2


Thống kê theo môn

Kết luận:

+ Môn Thong ke có điểm trung bình cao nhất (~9.2)
+ Một số môn có ít sinh viên → kết quả chưa thật sự đại diện

→ Nhận xét:

Có thể môn dễ hoặc sinh viên giỏi tập trung vào môn đó

# 4c. Phân loại điểm

In [6]:
def classify(score):
    if score >= 9:
        return "Xuat sac"
    elif score >= 6:
        return "Tot"
    else:
        return "Kem"

df["rank"] = df["score"].apply(classify)

print(df[["name","course_name","score","rank"]])

                name course_name  score      rank
0  Nguyen Minh Hoang   Giai tich    6.7       Tot
1       Tran Thi Lan    Thong ke    9.2  Xuat sac
2       Pham Van Nam   Giai tich    7.9       Tot
3      Bui Tien Dung    Thong ke    9.2  Xuat sac
4     Duong Huu Phuc   Giai tich    7.2       Tot
5       Cao Thi Hanh   Giai tich    7.0       Tot


Phân loại điểm

Kết luận:

Có đủ 3 nhóm:
+ Xuất sắc (≥ 9): ít nhưng nổi bật
+ Tốt (6–8.9): chiếm đa số
+ Kém (< 6): rất ít

→ Nhận xét:

Phần lớn sinh viên đạt mức khá–tốt → chất lượng chung ổn

# 5. Xếp hạng

In [7]:
# Toàn bộ
student["rank_all"] = student["score"].rank(ascending=False)

# Theo lớp 
student["rank_class"] = student.groupby("class")["score"].rank(ascending=False)

# Theo môn 
student["rank_course"] = student.groupby("course_id")["score"].rank(ascending=False)

# Top 3 cao & thấp
# Top 3 cao
top3 = student.sort_values("score", ascending=False).head(3)

# Top 3 thấp
bottom3 = student.sort_values("score", ascending=True).head(3)

print("Top 3 cao:\n", top3)
print("Top 3 thấp:\n", bottom3)

Top 3 cao:
    student_id           name     class  course_id  score  rank_all  \
1           2   Tran Thi Lan   Kinh Te       34.0    9.2       1.5   
6           7  Bui Tien Dung   Kinh Te       34.0    9.2       1.5   
2           3   Pham Van Nam  Toan Tin       12.0    7.9       3.0   

   rank_class  rank_course  
1         1.5          1.5  
6         1.5          1.5  
2         1.0          1.0  
Top 3 thấp:
    student_id               name     class  course_id  score  rank_all  \
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7       6.0   
9          10       Cao Thi Hanh  May Tinh       12.0    7.0       5.0   
8           9     Duong Huu Phuc   Kinh Te       12.0    7.2       4.0   

   rank_class  rank_course  
0         2.0          4.0  
9         1.0          3.0  
8         3.0          2.0  


- Toàn bộ

Kết luận:

+ Những sinh viên điểm ≥ 9 đứng đầu
+ Phản ánh chính xác năng lực tổng thể

- Theo lớp

Kết luận:

+ Có sự cạnh tranh trong từng lớp
+ Một số lớp có “top riêng” nhưng không cao toàn trường

- Theo môn

Kết luận:

+ Cho thấy sinh viên giỏi ở từng môn cụ thể
+ Có thể khác với xếp hạng toàn bộ

- Top 3 cao nhất & thấp nhất

Kết luận:

Top cao:
→ Nhóm sinh viên xuất sắc, cần được khen thưởng
Top thấp:
→ Nhóm cần hỗ trợ học tập


# 6. Thêm graduation_date

In [8]:
now = datetime.now()

# tạo số ngày chờ (điểm cao => ít ngày hơn)
student["days_to_grad"] = (10 - student["score"]) * 30

student["graduation_date"] = student["days_to_grad"].apply(lambda x: now + timedelta(days=x))

print(student[["name","score","graduation_date"]])

                name  score            graduation_date
0  Nguyen Minh Hoang    6.7 2026-07-11 09:46:50.640839
1       Tran Thi Lan    9.2 2026-04-27 09:46:50.640839
2       Pham Van Nam    7.9 2026-06-05 09:46:50.640839
6      Bui Tien Dung    9.2 2026-04-27 09:46:50.640839
8     Duong Huu Phuc    7.2 2026-06-26 09:46:50.640839
9       Cao Thi Hanh    7.0 2026-07-02 09:46:50.640839


Graduation_date

Kết luận:

+ Sinh viên điểm cao → tốt nghiệp sớm hơn
+ Sinh viên điểm thấp → tốt nghiệp muộn hơn

→ Nhận xét:

Đây là mô hình hợp lý để:
+ Khuyến khích học tốt
+ Phản ánh tiến độ học tập